In [25]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [26]:
load_dotenv()

True

In [27]:
model = ChatOpenAI(model='gpt-4o-mini')

In [28]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description='Detailed feedback for the essay')
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [29]:
structured_model = model.with_structured_output(EvaluationSchema)

In [30]:
essay = """Welcome to our blog on the "Rise of AI in India". In this blog, we will explore the history, current state, challenges, opportunities, case studies, and the future of artificial intelligence in India. AI has been gaining traction globally and India is no exception. With the increasing adoption of AI technologies across various sectors, India is poised to become a major player in the AI industry.

In the introduction, we will provide a definition of AI and give a brief overview of the rise of AI globally before delving into the specific rise of AI in India. We will discuss the early developments in AI research in India, the establishment of AI centers and research institutes, and the growth of the AI industry in the country.

Moving on to the current state of AI in India, we will explore the adoption of AI technologies by Indian businesses, the applications of AI in various sectors such as healthcare, agriculture, and finance, and the government initiatives to promote AI in the country. We will also discuss the challenges such as the skills gap in the AI workforce and ethical concerns, as well as the opportunities for innovation and growth in the AI industry.

We will present case studies of successful AI implementation in India, such as AI-powered diagnostics in healthcare, AI-driven crop monitoring in agriculture, and AI-based fraud detection in finance. These case studies will showcase the real-world impact of AI technologies in India.

Looking towards the future, we will discuss the potential for further growth and development of AI in India, the impact of AI on the Indian economy and job market, and the emerging trends in AI research and technology in the country.

In the conclusion, we will recap the key points discussed in the blog, share final thoughts on the rise of AI in India and its implications, and encourage readers to further explore the topic of AI in India. Stay tuned for an insightful journey into the world of AI in India."""

In [31]:
prompt = f'Evaluate the language quality of the following essay and provide and assign a score out of 10\ n {essay}'
structured_model.invoke(prompt).feedback

'The essay presents a structured and coherent overview of the topic, detailing the rise of AI in India effectively. The language used is clear and informative, though it lacks some flair that could engage the reader more. The use of formal language is appropriate for the subject matter, but there are instances where the writing could be made more concise. Additionally, while the outline of the blog is well-organized, it may benefit from more vivid examples or anecdotes to draw in readers and emphasize the significance of the content. Overall, the essay serves its purpose but could improve in engagement and expressiveness.'

In [41]:
class UPSCState(TypedDict):
    
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores : Annotated[list[int], operator.add]
    avg_score: float

In [42]:
def evaluate_language(state: UPSCState):
    prompt = f'Evaluate the langauge quality of the following essay and provide a feedback and assign a score out of 19 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [43]:
def evaluate_analysis(state: UPSCState):
    prompt = f'Evaluate the depth of the following essay and provide a feedback and assign a score out of 19 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [44]:
def evaluate_thought(state: UPSCState):
    prompt = f'Evaluate the clarity of though of the following essay and provide a feedback and assign a score out of 19 \n {state["essay"]}'

    output = structured_model.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [45]:
def final_evaluation(state: UPSCState):
    
    # summary feedback
    prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback - {state["language_feedback"]} \n depth of analysis - {state["analysis_feedback"]} \n clarity of thought feedback - {state["clarity_feedback"]}'

    overall_feedback = model.invoke(prompt).content

    # avg feedback
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}


In [46]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')


graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [47]:
initial_state = {
    'essay': essay
}

workflow.invoke(initial_state)

{'essay': 'Welcome to our blog on the "Rise of AI in India". In this blog, we will explore the history, current state, challenges, opportunities, case studies, and the future of artificial intelligence in India. AI has been gaining traction globally and India is no exception. With the increasing adoption of AI technologies across various sectors, India is poised to become a major player in the AI industry.\n\nIn the introduction, we will provide a definition of AI and give a brief overview of the rise of AI globally before delving into the specific rise of AI in India. We will discuss the early developments in AI research in India, the establishment of AI centers and research institutes, and the growth of the AI industry in the country.\n\nMoving on to the current state of AI in India, we will explore the adoption of AI technologies by Indian businesses, the applications of AI in various sectors such as healthcare, agriculture, and finance, and the government initiatives to promote AI 